# 🐟 Multiclass Fish Image Classification
**Models:** Custom CNN + VGG16 + ResNet50 + MobileNet + InceptionV3 + EfficientNetB0

### Steps:
1. Mount Google Drive & load dataset
2. Preprocess & augment images
3. Train all 6 models
4. Evaluate & compare models
5. Save the best model

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Install & Import Libraries

In [ ]:
!pip install -q tensorflow numpy pandas matplotlib seaborn scikit-learn pillow

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, InceptionV3, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Check GPU
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## Step 3 — Configuration

In [ ]:
# ── Dataset path (update this to match your Google Drive folder) ──
DATA_DIR  = '/content/drive/MyDrive/fish_image/data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'val')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

# ── Image & training settings ──
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 0.0001

# ── Folder to save trained models ──
MODEL_DIR = '/content/drive/MyDrive/fish_models'
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Class names ──
CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)

print(f'Classes ({NUM_CLASSES}):', CLASS_NAMES)

## Step 4 — Load & Augment Data

In [ ]:
# Training — with augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
)

# Validation & Test — only normalize
val_test_datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')

## Step 5 — Visualize Sample Images

In [ ]:
# Show a batch of sample training images
images, labels = next(train_gen)
short_names = [c.split()[-1] for c in CLASS_NAMES]

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(short_names[np.argmax(labels[i])], fontsize=8)
        ax.axis('off')
plt.suptitle('Sample Training Images (Augmented)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 6 — Model Definitions

In [ ]:
def build_cnn():
    """Custom CNN trained from scratch."""
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(*IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name='CNN')
    model.compile(optimizer=optimizers.Adam(LR),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def build_transfer_model(base_class, name):
    """Transfer learning model with frozen base + custom head."""
    base = base_class(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base.trainable = False  # Freeze base layers

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ], name=name)
    model.compile(optimizer=optimizers.Adam(LR),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model


# All models to train
MODEL_LIST = [
    ('CNN',            build_cnn),
    ('VGG16',          lambda: build_transfer_model(VGG16, 'VGG16')),
    ('ResNet50',       lambda: build_transfer_model(ResNet50, 'ResNet50')),
    ('MobileNet',      lambda: build_transfer_model(MobileNet, 'MobileNet')),
    ('InceptionV3',    lambda: build_transfer_model(InceptionV3, 'InceptionV3')),
    ('EfficientNetB0', lambda: build_transfer_model(EfficientNetB0, 'EfficientNetB0')),
]

print('Models ready:', [m[0] for m in MODEL_LIST])

## Step 7 — Train All Models

In [ ]:
training_histories = {}

for model_name, build_fn in MODEL_LIST:
    print(f'\n{"="*50}')
    print(f' Training: {model_name}')
    print(f'{"="*50}')

    model = build_fn()
    save_path = os.path.join(MODEL_DIR, f'{model_name}.h5')

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint(save_path, monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    training_histories[model_name] = history
    best_acc = max(history.history['val_accuracy'])
    print(f'  Best Val Accuracy: {best_acc:.4f}')
    print(f'  Saved to: {save_path}')

print('\n All models trained!')

## Step 8 — Plot Training History

In [ ]:
for model_name, history in training_histories.items():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title(f'{model_name} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend(); ax1.grid(True)

    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title(f'{model_name} — Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend(); ax2.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, f'{model_name}_history.png'), dpi=100)
    plt.show()

## Step 9 — Evaluate All Models on Test Set

In [ ]:
short_names = [c.split()[-1] for c in CLASS_NAMES]
all_results = {}

for model_name, _ in MODEL_LIST:
    model_path = os.path.join(MODEL_DIR, f'{model_name}.h5')
    if not os.path.exists(model_path):
        print(f'[SKIP] {model_name} — not found')
        continue

    print(f'\nEvaluating: {model_name}')
    model = tf.keras.models.load_model(model_path)

    test_gen.reset()
    preds = model.predict(test_gen, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    metrics = {
        'accuracy':  round(float(accuracy_score(y_true, y_pred)), 4),
        'precision': round(float(precision_score(y_true, y_pred, average='weighted', zero_division=0)), 4),
        'recall':    round(float(recall_score(y_true, y_pred, average='weighted', zero_division=0)), 4),
        'f1':        round(float(f1_score(y_true, y_pred, average='weighted', zero_division=0)), 4),
    }
    all_results[model_name] = metrics

    print(f'  Accuracy: {metrics["accuracy"]}  Precision: {metrics["precision"]}  Recall: {metrics["recall"]}  F1: {metrics["f1"]}')
    print(f'\n  Classification Report:\n')
    print(classification_report(y_true, y_pred, target_names=short_names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, f'{model_name}_confusion_matrix.png'), dpi=100)
    plt.show()

print('\n Evaluation complete!')

## Step 10 — Model Comparison

In [ ]:
# Print comparison table
df = pd.DataFrame(all_results).T
df.index.name = 'Model'
df = df.sort_values('accuracy', ascending=False)
print(df.to_string())

# Best model
best_model = df['accuracy'].idxmax()
print(f'\n Best Model: {best_model} (Accuracy: {df.loc[best_model, "accuracy"]:.4f})')

# Bar chart
metrics_list = ['accuracy', 'precision', 'recall', 'f1']
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
x = np.arange(len(all_results))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, color) in enumerate(zip(metrics_list, colors)):
    values = [all_results[m][metric] for m in all_results]
    ax.bar(x + i * width, values, width, label=metric.capitalize(), color=color)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(list(all_results.keys()), rotation=15)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'model_comparison.png'), dpi=100)
plt.show()

# Save results
df.to_csv(os.path.join(MODEL_DIR, 'evaluation_results.csv'))
with open(os.path.join(MODEL_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print('Results saved to Google Drive!')

## Step 11 — Save Best Model

In [ ]:
import shutil

best_model_path = os.path.join(MODEL_DIR, f'{best_model}.h5')
best_copy_path  = os.path.join(MODEL_DIR, 'best_model.h5')

shutil.copy(best_model_path, best_copy_path)
print(f'Best model ({best_model}) saved as: {best_copy_path}')

## Step 12 — Test Prediction on a Single Image

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

# Pick a random test image
test_class = CLASS_NAMES[0]
test_img_dir = os.path.join(TEST_DIR, test_class)
test_img_file = os.listdir(test_img_dir)[0]
test_img_path = os.path.join(test_img_dir, test_img_file)

# Load best model
best_model_loaded = tf.keras.models.load_model(best_copy_path)

# Preprocess
img = keras_image.load_img(test_img_path, target_size=IMG_SIZE)
img_array = keras_image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, 0)

# Predict
preds = best_model_loaded.predict(img_array)[0]
pred_class = CLASS_NAMES[np.argmax(preds)]
confidence = np.max(preds) * 100

# Show result
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f'True: {test_class.split()[-1]}\nPredicted: {pred_class.split()[-1]} ({confidence:.1f}%)', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()